# Valve Integrity Analytics — Notebook 3: Visualisations

**Inputs:**
- `valve_tests_enriched.csv` — full 780-row history with trend columns
- `integrity_summary_valve_level.csv` — 156-row current-state per well+valve
- `integrity_summary_well_level.csv` — 39-row rollup per well
- `exception_report.csv` — 2 critical wells

**Outputs:** 7 publication-quality charts saved to Drive

---
### Charts in this notebook
| # | Chart | Story it tells |
|---|---|---|
| 1 | Risk level distribution (stacked bar by asset) | Which asset has the most risk exposure? |
| 2 | Barrier status heatmap (wells × valve types) | Which well+valve combinations need attention? |
| 3 | Delta P vs Leak Rate scatter | How close are valves to their failure threshold? |
| 4 | Historical failure timeline | When did failures happen across the 2.5-year window? |
| 5 | Delta P trend lines (critical + high-risk wells) | Are problem wells getting worse over time? |
| 6 | Risk score distribution by valve type | Which valve type is riskiest overall? |
| 7 | Well-level risk scorecard (summary table chart) | One-page executive view of all 39 wells |

## Step 0 — Imports & settings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# ── Consistent colour palette across all charts ──
RISK_COLORS = {
    'CRITICAL' : '#D32F2F',
    'HIGH'     : '#F57C00',
    'MEDIUM'   : '#FBC02D',
    'LOW'      : '#388E3C'
}
BARRIER_COLORS = {
    'AT RISK'  : '#D32F2F',
    'MONITOR'  : '#F57C00',
    'HEALTHY'  : '#388E3C'
}
RESULT_COLORS = {'Failed': '#D32F2F', 'Passed': '#388E3C'}
ASSET_COLORS  = {
    'British Palm'       : '#1565C0',
    'European Yard'      : '#6A1B9A',
    'France Intial State': '#00695C'
}

plt.rcParams.update({
    'figure.dpi'       : 120,
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.titlesize'   : 12,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 10,
})

print('Setup complete.')

## Step 1 — Load all datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── All 4 CSVs are in outputs/ (saved there by Notebook 2) ──
BASE       = '/content/drive/MyDrive/valve_integrity_project/outputs/'
CHARTS_DIR = '/content/drive/MyDrive/valve_integrity_project/outputs/charts/'

import os
os.makedirs(CHARTS_DIR, exist_ok=True)

# Verify files exist before loading
for fname in ['valve_tests_enriched.csv', 'integrity_summary_valve_level.csv',
              'integrity_summary_well_level.csv', 'exception_report.csv']:
    status = '✅' if os.path.exists(BASE + fname) else '❌  NOT FOUND'
    print(f'{status}  {fname}')

enr   = pd.read_csv(BASE + 'valve_tests_enriched.csv',            parse_dates=['test_date'])
valve = pd.read_csv(BASE + 'integrity_summary_valve_level.csv',   parse_dates=['test_date'])
well  = pd.read_csv(BASE + 'integrity_summary_well_level.csv')
exc   = pd.read_csv(BASE + 'exception_report.csv',                parse_dates=['test_date'])

print(f'\nEnriched history : {enr.shape[0]:,} rows')
print(f'Valve summary    : {valve.shape[0]} rows  (one per well-valve)')
print(f'Well summary     : {well.shape[0]} rows  (one per well)')
print(f'Exception report : {exc.shape[0]} rows  (flagged wells only)')
print(f'\n✅ All loaded. Charts → {CHARTS_DIR}')

## Chart 1 — Risk Level Distribution by Asset
**Story:** Which asset has the most risk exposure across its valve portfolio?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Risk Level Distribution — Valve Portfolio', fontsize=13, fontweight='bold')

# ── Left: Stacked bar by asset ──
ax1 = axes[0]
risk_order = ['CRITICAL', 'MEDIUM', 'LOW']
risk_by_asset = (
    valve.groupby(['asset', 'risk_level'])
         .size()
         .unstack(fill_value=0)
         .reindex(columns=risk_order, fill_value=0)
)

bottom = np.zeros(len(risk_by_asset))
for level in risk_order:
    bars = ax1.bar(
        risk_by_asset.index,
        risk_by_asset[level],
        bottom=bottom,
        color=RISK_COLORS[level],
        label=level,
        edgecolor='white',
        linewidth=0.8
    )
    # Add count labels inside bars (only if segment is big enough)
    for bar, val in zip(bars, risk_by_asset[level]):
        if val > 2:
            ax1.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + bar.get_height() / 2,
                str(int(val)),
                ha='center', va='center',
                color='white', fontsize=9, fontweight='bold'
            )
    bottom += risk_by_asset[level].values

ax1.set_title('Valves by Risk Level per Asset')
ax1.set_ylabel('Number of Valves')
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=12)
ax1.legend(title='Risk Level', loc='upper right', fontsize=8)

# ── Right: Donut chart — overall portfolio ──
ax2 = axes[1]
overall = valve['risk_level'].value_counts()
# Keep order: CRITICAL → MEDIUM → LOW
ordered = {k: overall.get(k, 0) for k in ['CRITICAL', 'MEDIUM', 'LOW']}
colors  = [RISK_COLORS[k] for k in ordered.keys()]

wedges, texts, autotexts = ax2.pie(
    ordered.values(),
    labels=ordered.keys(),
    colors=colors,
    autopct='%1.0f%%',
    startangle=90,
    pctdistance=0.75,
    wedgeprops={'linewidth': 1.5, 'edgecolor': 'white'}
)
# Draw donut hole
centre_circle = plt.Circle((0, 0), 0.50, fc='white')
ax2.add_artist(centre_circle)
ax2.text(0, 0, f'{sum(ordered.values())}\nvalves', ha='center', va='center',
         fontsize=11, fontweight='bold', color='#333')
for at in autotexts:
    at.set_fontsize(9)
ax2.set_title('Overall Portfolio Risk')

plt.tight_layout()
plt.savefig(CHARTS_DIR + 'chart1_risk_distribution.png', bbox_inches='tight', dpi=140)
plt.show()
print('Chart 1 saved.')

## Chart 2 — Barrier Status Heatmap (Wells × Valve Types)
**Story:** A grid view — instantly see which cell in the well/valve matrix needs attention.

In [ ]:
# Pivot: rows = wells (sorted by risk), columns = valve types
status_map = {'HEALTHY': 0, 'MONITOR': 1, 'AT RISK': 2}

pivot = valve.pivot_table(
    index='well_id', columns='valve_type',
    values='barrier_status', aggfunc='first'
)
# Sort wells by max risk score descending so riskiest wells are at top
well_order = (
    valve.groupby('well_id')['risk_score']
         .max()
         .sort_values(ascending=True)  # ascending=True → riskiest at bottom for heatmap (origin bottom)
         .index.tolist()
)
pivot = pivot.reindex(well_order)
pivot_num = pivot.map(lambda x: status_map.get(x, -1) if pd.notna(x) else -1)

# Custom colormap: green → orange → red
cmap = mcolors.ListedColormap(['#388E3C', '#F57C00', '#D32F2F'])
bounds = [-0.5, 0.5, 1.5, 2.5]
norm   = mcolors.BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(8, 14))
im = ax.imshow(pivot_num.values, cmap=cmap, norm=norm, aspect='auto')

# Axis labels
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, fontsize=10, fontweight='bold')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=8)
ax.set_title('Barrier Status Heatmap — All Wells × All Valve Types', fontsize=12, fontweight='bold', pad=14)

# Add text annotations inside each cell
for i in range(pivot_num.shape[0]):
    for j in range(pivot_num.shape[1]):
        val = pivot.iloc[i, j]
        if pd.notna(val):
            short = {'HEALTHY': 'OK', 'MONITOR': 'MON', 'AT RISK': 'RISK'}.get(val, val)
            ax.text(j, i, short, ha='center', va='center',
                    color='white', fontsize=7, fontweight='bold')

# Legend patches
legend_patches = [
    mpatches.Patch(color='#388E3C', label='HEALTHY'),
    mpatches.Patch(color='#F57C00', label='MONITOR'),
    mpatches.Patch(color='#D32F2F', label='AT RISK'),
]
ax.legend(handles=legend_patches, loc='upper right',
          bbox_to_anchor=(1.18, 1.01), fontsize=9)

# Asset dividers: draw horizontal lines between assets
asset_order_map = valve.drop_duplicates('well_id').set_index('well_id')['asset']
prev_asset = None
for idx, wid in enumerate(well_order):
    curr = asset_order_map.get(wid)
    if curr != prev_asset and prev_asset is not None:
        ax.axhline(idx - 0.5, color='white', linewidth=2)
    prev_asset = curr

plt.tight_layout()
plt.savefig(CHARTS_DIR + 'chart2_barrier_heatmap.png', bbox_inches='tight', dpi=140)
plt.show()
print('Chart 2 saved.')

## Chart 3 — Delta P vs Leak Rate Scatter
**Story:** Every dot is one valve's latest test. Dots above the diagonal line = Failed. Dots close to the line = MONITOR.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Delta P vs Leak Rate — Latest Test per Valve', fontsize=13, fontweight='bold')

for ax, color_by in zip(axes, ['barrier_status', 'valve_type']):
    if color_by == 'barrier_status':
        color_map  = BARRIER_COLORS
        title_sfx  = 'coloured by Barrier Status'
    else:
        vt_palette = {'SCSSV':'#1565C0','PWV':'#6A1B9A','PMV':'#00695C','AMV':'#E65100'}
        color_map  = vt_palette
        title_sfx  = 'coloured by Valve Type'

    for category, grp in valve.groupby(color_by):
        ax.scatter(
            grp['leak_rate'], grp['delta_p'],
            c=color_map.get(category, '#999'),
            label=category, alpha=0.75, s=55, edgecolors='white', linewidth=0.5
        )

    # Diagonal: delta_p = leak_rate (the pass/fail boundary)
    max_val = max(valve['leak_rate'].max(), valve['delta_p'].max()) + 5
    diag    = np.linspace(0, max_val, 100)
    ax.plot(diag, diag, 'k--', linewidth=1, alpha=0.5, label='Failure threshold')

    # Shade the failure zone
    ax.fill_between(diag, diag, max_val, alpha=0.04, color='red')
    ax.text(max_val * 0.75, max_val * 0.88, 'FAIL zone',
            color='#D32F2F', fontsize=8, alpha=0.7)

    # Label the 2 critical wells
    for _, row in exc.iterrows():
        ax.annotate(
            f"{row['well_id']}\n{row['valve_type']}",
            xy=(row['leak_rate'], row['delta_p']),
            xytext=(row['leak_rate'] + 3, row['delta_p'] + 3),
            fontsize=7, color='#D32F2F',
            arrowprops=dict(arrowstyle='->', color='#D32F2F', lw=0.8)
        )

    ax.set_xlabel('Leak Rate (allowed max ΔP, PSI)')
    ax.set_ylabel('Actual Delta P (PSI)')
    ax.set_title(f'ΔP vs Leak Rate — {title_sfx}')
    ax.legend(fontsize=8, markerscale=1.1)
    ax.set_xlim(0, max_val)
    ax.set_ylim(0, max_val)

plt.tight_layout()
plt.savefig(CHARTS_DIR + 'chart3_delta_p_scatter.png', bbox_inches='tight', dpi=140)
plt.show()
print('Chart 3 saved.')

## Chart 4 — Historical Failure Timeline
**Story:** When across the 2.5-year window did failures happen? Are failures clustering in any period or asset?

In [ ]:
failures = enr[enr['result'] == 'Failed'].copy()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Historical Failure Timeline (2023–2026)', fontsize=13, fontweight='bold')

# ── Top: Event strip chart — each failure as a dot on the timeline ──
ax1 = axes[0]
asset_list = sorted(enr['asset'].unique())
y_map      = {a: i for i, a in enumerate(asset_list)}

for asset, grp in failures.groupby('asset'):
    ax1.scatter(
        grp['test_date'], [y_map[asset]] * len(grp),
        c=ASSET_COLORS[asset], s=100,
        zorder=3, label=asset, edgecolors='white', linewidth=0.8
    )
    # Label each dot with well_id + valve_type
    for _, row in grp.iterrows():
        ax1.annotate(
            f"{row['well_id']}·{row['valve_type']}",
            xy=(row['test_date'], y_map[asset]),
            xytext=(0, 10), textcoords='offset points',
            fontsize=6.5, ha='center', color=ASSET_COLORS[asset]
        )

ax1.set_yticks(list(y_map.values()))
ax1.set_yticklabels(list(y_map.keys()), fontsize=9)
ax1.set_title('Failure Events by Asset (each dot = one failed test)')
ax1.set_xlabel('')
ax1.set_xlim(enr['test_date'].min() - pd.Timedelta(days=30),
             enr['test_date'].max() + pd.Timedelta(days=30))
ax1.grid(axis='x', alpha=0.3)

# ── Bottom: Bar chart of failures per quarter ──
ax2 = axes[1]
failures['quarter'] = failures['test_date'].dt.to_period('Q').astype(str)
fail_by_quarter = failures.groupby(['quarter', 'asset']).size().unstack(fill_value=0)

x     = np.arange(len(fail_by_quarter))
width = 0.25
for i, asset in enumerate(asset_list):
    if asset in fail_by_quarter.columns:
        ax2.bar(
            x + i * width,
            fail_by_quarter[asset],
            width=width,
            color=ASSET_COLORS[asset],
            label=asset, edgecolor='white', linewidth=0.5
        )

ax2.set_xticks(x + width)
ax2.set_xticklabels(fail_by_quarter.index, rotation=30, ha='right', fontsize=8)
ax2.set_title('Failures per Quarter by Asset')
ax2.set_ylabel('Failure Count')
ax2.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(CHARTS_DIR + 'chart4_failure_timeline.png', bbox_inches='tight', dpi=140)
plt.show()
print('Chart 4 saved.')

## Chart 5 — Delta P Trend Lines (Critical + High-Risk Wells)
**Story:** For the wells that failed or have a rising trend, plot delta_p across all 5 test cycles to show deterioration.

In [ ]:
# Select wells worth showing trend lines for:
# 1) The 2 critical wells  2) EY-08 (2 failures)  3) Any with worsening trend
focus_wells = list(exc['well_id'].unique()) + ['EY-08', 'BP-10', 'FIS-16']
focus_wells = list(dict.fromkeys(focus_wells))  # deduplicate, preserve order

# Get all valve-level history for these wells from the enriched dataset
trend_data = enr[enr['well_id'].isin(focus_wells)].copy()

n_wells = len(focus_wells)
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=False)
fig.suptitle('Delta P Trend Over Time — Focus Wells', fontsize=13, fontweight='bold')
axes = axes.flatten()

vt_colors = {'SCSSV': '#1565C0', 'PWV': '#6A1B9A', 'PMV': '#00695C', 'AMV': '#E65100'}

for ax_idx, well in enumerate(focus_wells):
    ax  = axes[ax_idx]
    grp = trend_data[trend_data['well_id'] == well].sort_values('test_date')

    for vt, vt_grp in grp.groupby('valve_type'):
        vt_grp = vt_grp.sort_values('test_date')
        ax.plot(
            vt_grp['test_date'], vt_grp['delta_p'],
            marker='o', markersize=5,
            color=vt_colors[vt], label=vt, linewidth=1.5
        )
        # Mark failures with a red X
        failed = vt_grp[vt_grp['result'] == 'Failed']
        if not failed.empty:
            ax.scatter(
                failed['test_date'], failed['delta_p'],
                marker='X', color='#D32F2F', s=120, zorder=5, label='_nolegend_'
            )

    # Draw each valve's leak rate as a faint horizontal reference
    for vt, vt_grp in grp.groupby('valve_type'):
        lr = vt_grp['leak_rate'].iloc[-1]
        ax.axhline(lr, color=vt_colors[vt], linewidth=0.6, linestyle=':', alpha=0.5)

    # Asset label
    asset_name = grp['asset'].iloc[0] if not grp.empty else ''
    ax.set_title(f'{well}  |  {asset_name}', fontsize=9)
    ax.set_ylabel('Delta P (PSI)', fontsize=8)
    ax.tick_params(axis='x', rotation=25, labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    ax.legend(fontsize=7, loc='upper left')
    ax.text(0.98, 0.96, 'dotted = leak rate limit',
            transform=ax.transAxes, fontsize=6, ha='right', va='top', alpha=0.6)

# Hide any unused subplots
for i in range(n_wells, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.savefig(CHARTS_DIR + 'chart5_delta_p_trends.png', bbox_inches='tight', dpi=140)
plt.show()
print('Chart 5 saved.')

## Chart 6 — Risk Score Distribution by Valve Type
**Story:** Which valve type (SCSSV, PWV, PMV, AMV) tends to carry higher risk scores across the whole portfolio?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Risk Score Analysis by Valve Type', fontsize=13, fontweight='bold')

# ── Left: Box plot of risk score per valve type ──
ax1 = axes[0]
vt_order = ['SCSSV', 'PWV', 'PMV', 'AMV']
data_by_vt = [valve[valve['valve_type'] == vt]['risk_score'].values for vt in vt_order]

bp = ax1.boxplot(
    data_by_vt, labels=vt_order, patch_artist=True,
    medianprops=dict(color='white', linewidth=2)
)
box_colors = ['#1565C0', '#6A1B9A', '#00695C', '#E65100']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax1.set_title('Risk Score Distribution per Valve Type')
ax1.set_ylabel('Risk Score (0–100)')
ax1.set_xlabel('Valve Type')

# ── Right: Stacked bar — barrier status count per valve type ──
ax2 = axes[1]
bs_by_vt = (
    valve.groupby(['valve_type', 'barrier_status'])
         .size()
         .unstack(fill_value=0)
         .reindex(columns=['AT RISK', 'MONITOR', 'HEALTHY'], fill_value=0)
         .reindex(vt_order)
)

bottom = np.zeros(len(bs_by_vt))
for status in ['AT RISK', 'MONITOR', 'HEALTHY']:
    bars = ax2.bar(
        bs_by_vt.index, bs_by_vt[status],
        bottom=bottom,
        color=BARRIER_COLORS[status],
        label=status, edgecolor='white', linewidth=0.8
    )
    for bar, val in zip(bars, bs_by_vt[status]):
        if val > 1:
            ax2.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + bar.get_height() / 2,
                str(int(val)), ha='center', va='center',
                color='white', fontsize=9, fontweight='bold'
            )
    bottom += bs_by_vt[status].values

ax2.set_title('Barrier Status Count per Valve Type')
ax2.set_ylabel('Number of Valves')
ax2.legend(title='Barrier Status', fontsize=8)

plt.tight_layout()
plt.savefig(CHARTS_DIR + 'chart6_risk_by_valve_type.png', bbox_inches='tight', dpi=140)
plt.show()
print('Chart 6 saved.')

## Chart 7 — Well-Level Risk Scorecard
**Story:** A single executive view of all 39 wells — colour-coded by risk level with key metrics visible at a glance.

In [ ]:
# Sort wells: CRITICAL first → MEDIUM → LOW, then by score descending
level_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3}
well_sorted = well.copy()
well_sorted['_sort'] = well_sorted['well_risk_level'].map(level_order)
well_sorted = well_sorted.sort_values(['_sort', 'max_risk_score'], ascending=[True, False]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(13, 14))
ax.set_xlim(0, 10)
ax.set_ylim(-0.5, len(well_sorted) - 0.5)
ax.axis('off')
ax.set_title('Well Integrity Risk Scorecard — All Assets', fontsize=13, fontweight='bold', pad=16)

# Column header positions
cols = {
    'Well'         : 0.5,
    'Asset'        : 2.2,
    'Risk Score'   : 4.3,
    'Risk Level'   : 5.4,
    'Worst Barrier': 6.7,
    'Worst Valve'  : 8.1,
    'Failures'     : 9.3
}

# Header row
header_y = len(well_sorted) - 0.1
for col_name, x in cols.items():
    ax.text(x, header_y, col_name, fontsize=8, fontweight='bold', color='#333', va='bottom')
ax.axhline(len(well_sorted) - 0.3, color='#999', linewidth=0.8)

# Merge failure count info
well_failures = enr.groupby('well_id')['result'].apply(
    lambda x: (x == 'Failed').sum()
).reset_index()
well_failures.columns = ['well_id', 'hist_failures']
well_sorted = well_sorted.merge(well_failures, on='well_id', how='left')

for i, row in well_sorted.iterrows():
    y     = len(well_sorted) - 1 - i
    level = row['well_risk_level']
    color = RISK_COLORS.get(level, '#888')

    # Row background
    bg_color = {'CRITICAL': '#FFEBEE', 'HIGH': '#FFF3E0',
                'MEDIUM': '#FFFDE7', 'LOW': '#F1F8E9'}.get(level, '#FAFAFA')
    ax.barh(y, 10, left=0, height=0.82, color=bg_color, alpha=0.6, zorder=1)

    # Row data
    barrier_color = BARRIER_COLORS.get(row['worst_barrier_status'], '#888')

    ax.text(cols['Well'],          y, row['well_id'],                    fontsize=8, va='center', fontweight='bold')
    ax.text(cols['Asset'],         y, row['asset'],                      fontsize=7.5, va='center', color='#444')
    ax.text(cols['Risk Score'],    y, str(int(row['max_risk_score'])),   fontsize=9, va='center', fontweight='bold', color=color)
    ax.text(cols['Risk Level'],    y, level,                             fontsize=7.5, va='center', color=color, fontweight='bold')
    ax.text(cols['Worst Barrier'], y, row['worst_barrier_status'],       fontsize=7.5, va='center', color=barrier_color, fontweight='bold')
    ax.text(cols['Worst Valve'],   y, str(row['worst_valve']),           fontsize=8, va='center', color='#333')
    ax.text(cols['Failures'],      y, str(int(row['hist_failures'])),    fontsize=8, va='center',
            color='#D32F2F' if row['hist_failures'] > 0 else '#388E3C', fontweight='bold')

    # Light divider line
    ax.axhline(y - 0.42, color='#ddd', linewidth=0.4, zorder=0)

# Asset section labels in right margin
prev_asset = None
for i, row in well_sorted.iterrows():
    y = len(well_sorted) - 1 - i
    if row['asset'] != prev_asset:
        ax.text(9.95, y, row['asset'], fontsize=6.5, va='center',
                ha='right', color=ASSET_COLORS.get(row['asset'], '#888'),
                style='italic')
        prev_asset = row['asset']

plt.tight_layout()
plt.savefig(CHARTS_DIR + 'chart7_well_scorecard.png', bbox_inches='tight', dpi=140)
plt.show()
print('Chart 7 saved.')

## Final Summary — All Charts Generated

In [ ]:
import os

charts = sorted([f for f in os.listdir(CHARTS_DIR) if f.endswith('.png')])
print(f'Charts saved to: {CHARTS_DIR}')
print(f'Total charts   : {len(charts)}')
for c in charts:
    size_kb = os.path.getsize(CHARTS_DIR + c) // 1024
    print(f'  {c}  ({size_kb} KB)')

print()
print('Key findings from this notebook:')
print(f"  - {(valve['risk_level'] == 'CRITICAL').sum()} valves at CRITICAL risk (latest test Failed)")
print(f"  - {(valve['barrier_status'] == 'MONITOR').sum()} valves on MONITOR (passed but close to limit)")
print(f"  - {(valve['rising_dp_trend'] == True).sum()} valves showing a rising delta_p trend")
print(f"  - {len(exc)} wells in the exception report requiring engineer review")
print()
print('Next -> Notebook 4: PDF well dossier generation using reportlab')

## Notebook 3 Summary

| Chart | File | Story |
|---|---|---|
| 1 | `chart1_risk_distribution.png` | Risk levels by asset + portfolio donut |
| 2 | `chart2_barrier_heatmap.png` | Well × valve status grid |
| 3 | `chart3_delta_p_scatter.png` | Every valve plotted against its failure threshold |
| 4 | `chart4_failure_timeline.png` | When failures happened across 2.5 years |
| 5 | `chart5_delta_p_trends.png` | Deterioration curves for focus wells |
| 6 | `chart6_risk_by_valve_type.png` | Which valve type carries most risk? |
| 7 | `chart7_well_scorecard.png` | Executive one-pager for all 39 wells |

---
**Next → Notebook 4:** PDF well dossier — one-page PDF per critical well using `reportlab`.